## DEMO 2.3: **Naive Bayes Classification**
<u>Nội dung</u>:
https://scikit-learn.org/stable/modules/naive_bayes.html
1. Gaussian NB
2. Multinomial NB

<u>Cập nhật</u>: **04/2026**




> Bản hoàn chỉnh: chạy được cả trên Colab lẫn máy cục bộ (tự dùng dữ liệu của sklearn / dữ liệu mô phỏng nếu không có file), thay phần nhập tay bằng hàm, và **bổ sung hai phần còn bỏ trống**: *So sánh Gaussian NB và Multinomial NB* và *Tiền xử lý DL (rời rạc hóa)*.

---
#### **MÔI TRƯỜNG TRIỂN KHAI ỨNG DỤNG**
---

In [ ]:
## Kết nối Google Drive (nếu chạy trên Colab); chạy cục bộ thì dùng thư mục ./
import os
try:
    from google.colab import drive
    drive.mount("/content/gdrive")
    folder = '/content/gdrive/My Drive/Edu/1. UEH/Machine Learning/1. Demo/Ch2 - Supervised Learning'
except ImportError:
    folder = '.'                       # chạy cục bộ: đặt thư mục Data/ cạnh notebook

In [ ]:
## Thư viện xử lý DL
import numpy                    as np
import pandas                   as pd

## Xây dựng mô hình
import joblib                   as jlb
from   sklearn.model_selection  import train_test_split
from   sklearn.naive_bayes      import GaussianNB, BernoulliNB, MultinomialNB

## Biểu diễn trực quan DL
import matplotlib.pyplot        as plt
import seaborn                  as sbn

## Thư viện khác
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import confusion_matrix, classification_report

**Nhắc nhanh – KHI NÀO DÙNG biến thể nào**
| Mô hình | Dữ liệu phù hợp |
|---|---|
| Gaussian NB | features **liên tục**, xấp xỉ chuông trong mỗi lớp (đo đạc: chiều dài, cân nặng…) |
| Multinomial NB | features **đếm/tần suất không âm** (số lần xuất hiện từ, số lần mua…) |
| Bernoulli NB | features **nhị phân** (có/không) |

---
#### 1. **Gaussian NB**
---

In [ ]:
## Tập tin dữ liệu IRIS (đọc từ Data/Iris.xls nếu có, ngược lại dùng bộ Iris của sklearn)
p = folder + '/Data/Iris.xls'
if os.path.exists(p):
    data = pd.read_excel(p)
else:
    from sklearn.datasets import load_iris
    _ir = load_iris(as_frame=True)
    data = _ir.data.copy(); data.columns = ['sepallength', 'sepalwidth', 'petallength', 'petalwidth']
    data['species'] = _ir.target_names[_ir.target]

In [ ]:
## Các features
X = data.drop('species', axis = 1)
X.head()

## Biến target: species
y = data.species

print(pd.concat([X, y], axis = 1).head())

## Chia tập dữ liệu thành training, test sets theo tỷ lệ 80:20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = .2, random_state = 1)

In [ ]:
##------------------------------------------------------------------------------
## Huấn luyện mô hình Gaussian NB
##------------------------------------------------------------------------------
Gmodel = GaussianNB()
Gmodel.fit(X_train, y_train)

In [ ]:
## Khai thác mô hình Gaussian NB (nhập tay – đóng gói thành hàm để notebook chạy tự động được)
def du_doan_iris(model, spl, spw, ptl, ptw):
    X_new = pd.DataFrame([[spl, spw, ptl, ptw]], columns=X.columns)
    return model.predict(X_new)[0]

def nhap_tay_iris(model):
    tiep = 'C'
    while (tiep.upper() == 'C'):
        spl = float(eval(input(f'sepallength [4.3, 7.9]: ')))
        spw = float(eval(input(f'sepalwidth  [2.0, 4.4]: ')))
        ptl = float(eval(input(f'petallength [1.0, 6.9]: ')))
        ptw = float(eval(input(f'petalwidth  [0.1, 2.5]: ')))
        print(f'\nMẫu {[spl, spw, ptl, ptw]} được dự đoán là {du_doan_iris(model, spl, spw, ptl, ptw)}\n')
        tiep = input('Tiếp tục (C/K) ? ')

# Chạy thử vài mẫu (bỏ comment dòng cuối để nhập tay)
for m in [(5.1, 3.5, 1.4, 0.2), (6.0, 2.9, 4.5, 1.5), (6.9, 3.1, 5.4, 2.1)]:
    print(f'Mẫu {m} được dự đoán là {du_doan_iris(Gmodel, *m)}')
# nhap_tay_iris(Gmodel)

In [ ]:
## Số trường hợp đoán ĐÚNG trong tập Test
df = pd.DataFrame({'yHat':Gmodel.predict(X_test), 'y': pd.Series(y_test)})
print(f'Accuracy (Test set) = {len(df[df.yHat == df.y])} / {len(X_test)}')

In [ ]:
## Ma trận nhầm lẫn và báo cáo chi tiết (Test set) – Gaussian NB trên Iris
y_hat_g = Gmodel.predict(X_test)
print(confusion_matrix(y_test, y_hat_g)); print(classification_report(y_test, y_hat_g, digits=3))
jlb.dump(Gmodel, 'Gmodel_NB.joblib')     # lưu mô hình (dùng lại bằng jlb.load)

---
## 2. **Multinomial NB**
---

In [ ]:
## Tập tin dữ liệu Social_Network_Ads (đọc từ Data/ nếu có, ngược lại sinh dữ liệu mô phỏng cùng cấu trúc)
p = folder + '/Data/Social_Network_Ads.csv'
if os.path.exists(p):
    data = pd.read_csv(p)
else:
    rng = np.random.default_rng(1); n = 400
    age = rng.integers(18, 60, n); sal = rng.integers(15000, 150000, n)
    z = 0.25 * (age - 38) + 0.00004 * (sal - 70000) + rng.normal(0, 1, n) - 0.3
    data = pd.DataFrame({'User ID': np.arange(15600000, 15600000 + n), 'Gender': rng.choice(['Male', 'Female'], n),
                         'Age': age, 'EstimatedSalary': sal, 'EstimatedSalary_K': sal / 1000, 'Purchased': (z > 0).astype(int)})
data.head()

In [ ]:
## Các features: Gender, Age, EstimatedSalary
X = data.drop(['User ID', 'EstimatedSalary_K', 'Purchased'], axis = 1, errors = 'ignore')

## Chuyển Gender thành kiểu Numerical
sex = {'Male':1, 'Female':0}
X['Male'] = [sex[i] for i in data.Gender]
X         = X.drop(['Gender'], axis = 1)

## Biến target: Purchased
y = data.Purchased

print(pd.concat([X, y], axis = 1).head())

## Chia tập dữ liệu thành training, test sets theo tỷ lệ 80:20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = .2, random_state = 1)

In [ ]:
##------------------------------------------------------------------------------
## Huấn luyện mô hình Multinomial NB
##------------------------------------------------------------------------------
Mmodel = MultinomialNB()
Mmodel.fit(X_train, y_train)

In [ ]:
## Khai thác mô hình Multinomial NB (đóng gói thành hàm)
labels = np.array(['KHÔNG mua', 'MUA'])

def du_doan_ads(model, age, sal, male):
    X_new = pd.DataFrame([[age, sal, male]], columns=X.columns)       # thứ tự cột: Age, EstimatedSalary, Male
    return labels[model.predict(X_new)[0]]

def nhap_tay_ads(model):
    tiep = 'C'
    while (tiep.upper() == 'C'):
        age = float(eval(input('Tuổi:             ')))
        sal = float(eval(input('Mức lương:        ')))
        gt  = float(eval(input('Nam (1) / Nữ (0): ')))
        print('Mẫu', [age, sal, gt], 'được dự đoán là', du_doan_ads(model, age, sal, gt))
        tiep = input('Tiếp tục (C/K) ? ')

for m in [(25, 30000, 0), (45, 90000, 1), (35, 60000, 1)]:
    print('Mẫu', m, 'được dự đoán là', du_doan_ads(Mmodel, *m))
# nhap_tay_ads(Mmodel)

In [ ]:
print('Multinomial NB (dữ liệu thô): Accuracy train = %.4f | test = %.4f' % (Mmodel.score(X_train, y_train), Mmodel.score(X_test, y_test)))
print('Phân bố lớp:', y.value_counts(normalize=True).round(3).to_dict())

---
## 3. So sánh Gaussian NB và Multinomial NB
Hai bộ dữ liệu: **Iris** (features liên tục, dạng chuông) và **Social_Network_Ads** (tuổi, lương, giới tính – thang đo khác nhau).

**Kỳ vọng:** Gaussian NB phù hợp dữ liệu liên tục; Multinomial NB giả định dữ liệu dạng đếm nên thường kém trên dữ liệu liên tục chưa xử lý.

In [ ]:
def so_sanh_nb(X, y, ten, test_size=.2, seed=1):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y)
    kq = {}
    for tm, m in [('GaussianNB', GaussianNB()), ('MultinomialNB', MultinomialNB()), ('BernoulliNB (mặc định, ngưỡng 0)', BernoulliNB())]:
        m.fit(Xtr, ytr); kq[tm] = {'Acc_train': m.score(Xtr, ytr), 'Acc_test': m.score(Xte, yte)}
    print(f'=== {ten} ==='); print(pd.DataFrame(kq).T.round(4)); print()

# Iris
X_iris, y_iris = data_iris = None, None
_iris = pd.read_excel(folder + '/Data/Iris.xls') if os.path.exists(folder + '/Data/Iris.xls') else None
if _iris is None:
    from sklearn.datasets import load_iris
    _ir = load_iris(as_frame=True); X_iris, y_iris = _ir.data, _ir.target_names[_ir.target]
else:
    X_iris, y_iris = _iris.drop('species', axis=1), _iris.species
so_sanh_nb(X_iris, y_iris, 'Iris (dữ liệu thô)')

# Social Network Ads (dùng lại X, y đã chuẩn bị ở phần 2)
so_sanh_nb(X, y, 'Social_Network_Ads (dữ liệu thô)')

**Nhận xét (từ lần chạy thử):** trên Iris, MultinomialNB trên dữ liệu thô vẫn khá tốt vì các feature đều dương và các lớp khác nhau rõ ở mức độ lớn của giá trị (nhưng đó là "may mắn" của dữ liệu, không phải do giả định đúng); BernoulliNB mặc định chỉ đạt ~33% vì ngưỡng 0 làm mọi feature = 1. Trên Social_Network_Ads, Gaussian NB vượt trội (~86–90%) còn Multinomial/Bernoulli chỉ ~50%, xấp xỉ đoán lớp đa số, vì `EstimatedSalary` có thang đo rất lớn áp đảo các feature khác và không phải dữ liệu đếm. Cần tiền xử lý (phần 4).

---
## 4. Tiền xử lý DL (rời rạc hóa)
**Rời rạc hóa** (binning) chuyển feature liên tục thành các khoảng 0, 1, 2, …, B−1 để dùng được với **MultinomialNB** (dữ liệu đếm/hạng mục) hoặc **BernoulliNB** (nhị phân).

**KHI NÀO DÙNG:** muốn dùng Multinomial/Bernoulli/Categorical NB cho dữ liệu liên tục, hoặc dữ liệu liên tục lệch/không chuông (Gaussian giả định sai). Chia theo **quantile** (số mẫu mỗi khoảng đều nhau) khi dữ liệu lệch; chia **uniform** khi muốn khoảng cách đều. Luôn `fit` bộ rời rạc hóa trên **train** rồi `transform` test.

In [ ]:
def rời_rac_hoa_va_so_sanh(X, y, ten, so_khoang=(3, 5, 8, 10), chien_luoc='quantile', seed=1):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, random_state=seed, stratify=y)
    gm = GaussianNB().fit(Xtr, ytr); rows = [{'Cách xử lý': 'Gaussian (dữ liệu thô)', 'Acc_train': gm.score(Xtr, ytr), 'Acc_test': gm.score(Xte, yte)}]
    for b in so_khoang:
        kb = KBinsDiscretizer(n_bins=b, encode='ordinal', strategy=chien_luoc).fit(Xtr)     # fit trên TRAIN
        Dtr, Dte = kb.transform(Xtr), kb.transform(Xte)
        mm = MultinomialNB().fit(Dtr, ytr)
        rows.append({'Cách xử lý': f'Multinomial, {b} khoảng ({chien_luoc})', 'Acc_train': mm.score(Dtr, ytr), 'Acc_test': mm.score(Dte, yte)})
    # Cách đúng hơn cho Multinomial: one-hot các khoảng (mỗi khoảng là 1 'từ'), hoặc CategoricalNB cho feature hạng mục
    b = 5
    kb1 = KBinsDiscretizer(n_bins=b, encode='onehot-dense', strategy=chien_luoc).fit(Xtr)
    O_tr, O_te = kb1.transform(Xtr), kb1.transform(Xte); mo = MultinomialNB().fit(O_tr, ytr)
    rows.append({'Cách xử lý': f'Multinomial + one-hot, {b} khoảng', 'Acc_train': mo.score(O_tr, ytr), 'Acc_test': mo.score(O_te, yte)})
    kb2 = KBinsDiscretizer(n_bins=b, encode='ordinal', strategy=chien_luoc).fit(Xtr)
    K_tr, K_te = kb2.transform(Xtr).astype(int), kb2.transform(Xte).astype(int)
    cn = CategoricalNB(min_categories=b).fit(K_tr, ytr)
    rows.append({'Cách xử lý': f'CategoricalNB, {b} khoảng', 'Acc_train': cn.score(K_tr, ytr), 'Acc_test': cn.score(K_te, yte)})
    # Bernoulli: nhị phân hóa theo trung vị từng feature
    med = np.median(np.asarray(Xtr, float), axis=0)
    Btr, Bte = (np.asarray(Xtr, float) > med).astype(int), (np.asarray(Xte, float) > med).astype(int)
    bm = BernoulliNB(binarize=None).fit(Btr, ytr)
    rows.append({'Cách xử lý': 'Bernoulli (nhị phân hóa theo trung vị)', 'Acc_train': bm.score(Btr, ytr), 'Acc_test': bm.score(Bte, yte)})
    print(f'=== {ten} ==='); print(pd.DataFrame(rows).set_index('Cách xử lý').round(4)); print()

rời_rac_hoa_va_so_sanh(X_iris, y_iris, 'Iris')
rời_rac_hoa_va_so_sanh(X, y, 'Social_Network_Ads')

In [ ]:
## Minh họa trực quan: phân bố Age/EstimatedSalary trước và sau khi rời rạc hóa (5 khoảng quantile)
kb = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile').fit(X[['Age', 'EstimatedSalary']])
D = pd.DataFrame(kb.transform(X[['Age', 'EstimatedSalary']]), columns=['Age_bin', 'Salary_bin'])
fig, ax = plt.subplots(2, 2, figsize=(10, 6))
X['Age'].hist(ax=ax[0, 0], bins=20); ax[0, 0].set_title('Age (thô)')
X['EstimatedSalary'].hist(ax=ax[0, 1], bins=20); ax[0, 1].set_title('EstimatedSalary (thô)')
D['Age_bin'].value_counts().sort_index().plot.bar(ax=ax[1, 0], title='Age (rời rạc 5 khoảng)')
D['Salary_bin'].value_counts().sort_index().plot.bar(ax=ax[1, 1], title='EstimatedSalary (rời rạc 5 khoảng)')
plt.tight_layout(); plt.show()
print('Các mốc cắt (quantile):'); print({c: np.round(e, 1).tolist() for c, e in zip(['Age', 'EstimatedSalary'], kb.bin_edges_)})

**Kết luận (từ lần chạy thử; con số có thể khác với dữ liệu thật của bạn)**
- Dữ liệu **liên tục dạng chuông** (Iris) → **Gaussian NB** dùng trực tiếp, thường là lựa chọn tốt nhất; rời rạc hóa rồi dùng Multinomial không giúp ích (mất thông tin).
- **Chỉ đánh số khoảng (ordinal) rồi đưa vào MultinomialNB là chưa đúng bản chất**: chỉ số khoảng 0,1,2… bị hiểu như "số lần xuất hiện", tức ngầm giả định thứ tự và độ lớn. Cách đúng hơn là **one-hot** các khoảng (Multinomial) hoặc dùng **CategoricalNB**; **Bernoulli sau khi nhị phân hóa theo trung vị** cũng cải thiện mạnh so với ngưỡng mặc định 0.
- Số khoảng ít → mất thông tin; quá nhiều → thưa, dễ overfit; chọn bằng validation/cross-validation.
- Lỗi thường gặp: fit bộ rời rạc hóa trên toàn bộ dữ liệu (rò rỉ test), và dùng MultinomialNB với giá trị âm (báo lỗi).